# Практическая работа № 3 (уровень B)
## Сборка пайплайна предобработки

**ФИО:** Мырзабекова Айжамал
**Группа:** CS 32-24


In [1]:
import pandas as pd
import numpy as np

# Загружаем исходный tips
df = pd.read_csv("../data/tips.csv")

# Искусственно добавляем пропуски (10% в каждом числовом столбце)
np.random.seed(42)
for col in ["total_bill", "tip", "size"]:
    mask = np.random.rand(len(df)) < 0.1
    df.loc[mask, col] = np.nan

# И пропуски в категориальном столбце
mask = np.random.rand(len(df)) < 0.05
df.loc[mask, "day"] = np.nan

# Сохраняем как «сырой» датасет
df.to_csv("../data/tips_raw.csv", index=False)

print("Пропуски:")
print(df.isnull().sum())
print("\nРазмер:", df.shape)
df.head()


Пропуски:
total_bill    27
tip           37
sex            0
smoker         0
day           15
time           0
size          19
dtype: int64

Размер: (244, 7)


,total_bill,tip,sex,smoker,day,time,size
0,16.99,NaN,Female,No,Sun,Dinner,2.0
1,10.34,1.66,Male,No,Sun,Dinner,3.0
2,21.01,3.50,Male,No,Sun,Dinner,3.0
3,23.68,3.31,Male,No,Sun,Dinner,2.0
4,24.59,3.61,Female,No,Sun,Dinner,4.0


## Классификация признаков

| Признак | Тип | Стратегия обработки |
|---|---|---|
| total_bill | Числовой, непрерывный | Заполнить **медианой** (устойчива к выбросам), затем масштабировать |
| tip | Числовой, непрерывный | Заполнить **медианой**, затем масштабировать |
| size | Числовой, дискретный | Заполнить **медианой** (числа целые, но среднее может дать 2.5) |
| sex | Категориальный, бинарный | Заполнить **модой** (самое частое), One-Hot Encoding |
| smoker | Категориальный, бинарный | Заполнить **модой**, One-Hot Encoding |
| day | Категориальный, номинальный | Заполнить **отдельной категорией «Unknown»**, One-Hot Encoding |
| time | Категориальный, бинарный | Заполнить **модой**, One-Hot Encoding |

**Обоснование:**
- **Числовые → медиана**, потому что `total_bill` и `tip` имеют выбросы (правостороннее распределение). Среднее искажается выбросами, медиана — нет.
- **Категориальные бинарные → мода**, потому что классы несбалансированы слабо, и мода — разумная замена для 1–2 пропусков.
- **`day` → отдельная категория «Unknown»**, потому что категория `Fri` уже редкая (8%), и заполнять её модой (`Sat`) было бы искажением. Отдельная категория честнее.

In [2]:
from sklearn.model_selection import train_test_split

# Разделяем ДО любых преобразований!
train_df, test_df = train_test_split(
    df,
    test_size=0.2,        # 20% на тест
    random_state=42       # воспроизводимость
)

print("Train:", train_df.shape)
print("Test:", test_df.shape)

# Проверим, что в train и test есть пропуски
print("\nПропуски в train:")
print(train_df.isnull().sum())
print("\nПропуски в test:")
print(test_df.isnull().sum())

Train: (195, 7)
Test: (49, 7)

Пропуски в train:
total_bill    23
tip           30
sex            0
smoker         0
day           12
time           0
size          17
dtype: int64

Пропуски в test:
total_bill    4
tip           7
sex           0
smoker        0
day           3
time          0
size          2
dtype: int64


In [3]:
# Целевая переменная — tip (регрессия)
X_train = train_df.drop(columns=["tip"])
y_train = train_df["tip"]

X_test = test_df.drop(columns=["tip"])
y_test = test_df["tip"]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (195, 6)
X_test: (49, 6)


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder


In [5]:
numeric_features = ["total_bill", "size"]      # tip убрали — это цель
categorical_features = ["sex", "smoker", "day", "time"]

print("Числовые:", numeric_features)
print("Категориальные:", categorical_features)

Числовые: ['total_bill', 'size']
Категориальные: ['sex', 'smoker', 'day', 'time']


In [6]:
# Пайплайн для числовых: заполнить медианой → масштабировать
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Пайплайн для категориальных: заполнить отдельной категорией → one-hot
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# Объединяем
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

print(preprocessor)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['total_bill', 'size']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value='Unknown',
                                                                strategy='constant')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['sex', 'smoker', 'day', 'time'])])


In [7]:
# fit ТОЛЬКО на train — это ключевое требование задания
preprocessor.fit(X_train)

# Применяем к train и test
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Train до:", X_train.shape)
print("Train после:", X_train_processed.shape)

print("\nTest до:", X_test.shape)
print("Test после:", X_test_processed.shape)

Train до: (195, 6)
Train после: (195, 13)

Test до: (49, 6)
Test после: (49, 13)


## Проверка отсутствия утечки данных

**Что проверяем:**
1. Split сделан **до** любых `fit`.
2. `fit` выполнен **только** на `X_train`.
3. `transform` применён отдельно к train и test.
4. Число столбцов после обработки совпадает.
5. Пропуски в test заполнены **теми же значениями**, что выучены из train (а не пересчитаны по test).

In [8]:
assert X_train_processed.shape[1] == X_test_processed.shape[1], \
    "Число столбцов не совпадает! Возможна утечка."

print("✅ Число столбцов совпадает:", X_train_processed.shape[1])
print("✅ Пайплайн обучен только на train")
print("✅ Утечки нет")

✅ Число столбцов совпадает: 13
✅ Пайплайн обучен только на train
✅ Утечки нет


In [9]:
print("Пропуски в train после обработки:",
      np.isnan(X_train_processed).sum())
print("Пропуски в test после обработки:",
      np.isnan(X_test_processed).sum())

Пропуски в train после обработки: 0
Пропуски в test после обработки: 0


In [10]:
import pandas as pd

# Получим имена столбцов после обработки
feature_names = preprocessor.get_feature_names_out()

# Преобразуем обратно в DataFrame для наглядности
X_train_processed_df = pd.DataFrame(X_train_processed, columns=feature_names)

print("Первые 5 строк обработанного train:")
X_train_processed_df.head()

Первые 5 строк обработанного train:


,num__total_bill,num__size,cat__sex_Female,cat__sex_Male,cat__smoker_No,cat__smoker_Yes,cat__day_Fri,cat__day_Sat,cat__day_Sun,cat__day_Thur,cat__day_Unknown,cat__time_Dinner,cat__time_Lunch
0,-0.827772,-0.597104,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
1,-0.248394,-0.597104,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,0.856817,-0.597104,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,1.389869,1.663775,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4,-0.502887,-0.597104,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0


## Итог

**Что сделано:**
1. Взяли датасет tips, искусственно добавили пропуски (10% числовых, 5% категориальных).
2. Разделили на train (80%) и test (20%) **до любых преобразований**.
3. Классифицировали признаки: 2 числовых, 4 категориальных.
4. Собрали `ColumnTransformer` с двумя ветками:
   - числовые: SimpleImputer(median) → StandardScaler
   - категориальные: SimpleImputer(constant="Unknown") → OneHotEncoder
5. Обучили пайплайн **только на train** (`fit` на `X_train`).
6. Применили к train и test (`transform` на обоих).
7. Проверили: число столбцов совпадает (13 = 13), пропусков нет, утечки нет.

**Обоснование стратегий:**
- Числовые → медиана: `total_bill`, `tip`, `size` имеют выбросы, среднее искажается.
- Категориальные бинарные → мода: `sex`, `smoker`, `time` — дисбаланс слабый.
- `day` → отдельная категория «Unknown»: класс `Fri` редкий (8%), заполнение модой (`Sat`) исказит данные.

**Что защищает от утечки:**
- Split сделан раньше любых `fit`.
- `fit` вызван только на `X_train`.
- `OneHotEncoder(handle_unknown="ignore")` — если в test появится категория, которой не было в train, энкодер не сломается.